
# Credit Card Transactions — Databricks Data Engineering Project

## Project Overview

This project builds a production-oriented data pipeline for credit card transaction data using Databricks and Delta Lake.

The pipeline follows a layered architecture:

**Bronze → Silver → Gold**

- **Bronze:** raw source data
- **Silver:** cleaned and standardized transaction data
- **Gold:** curated analytical data products for user, card, merchant, and fraud analysis

Authoritative Silver table:

`credit_card_project.silver.cleaned_transactions`

Silver population:

**24,386,900 transactions**

**Gold v1 is complete, validated, engineering-reviewed, committed, and pushed to `main`.**



# Architecture

Raw Transaction Data ───► Bronze ───► Silver
   
cleaned_transactions

24,386,900 rows

──► `gold_user_behavior`

──► `gold_card_behavior`

──► `gold_merchant_behavior`

──► `gold_fraud_features`



# Gold v1 Data Products

## 1. gold_user_behavior

**Grain:** one row per `user_id`

**Rows:** 2,000

Provides user-level behavioral metrics including:

- transaction volume
- total spend
- average transaction amount
- fraud count and fraud rate
- unique cards, merchants, and MCCs
- online transaction percentage
- weekend transaction percentage
- nighttime transaction percentage


## 2. gold_card_behavior

**Grain:** one row per `(user_id, card_id)`

**Rows:** 6,146

The population is driven by Bronze cards, with Silver transaction aggregates left joined onto the Bronze population.

This preserves cards that currently have no Silver transactions.

Key characteristics:

- Bronze is the population driver.
- Debit (Prepaid) remains distinct from ordinary Debit.
- Account opening and expiry values remain in their original MM/YYYY representation.
- Percentage metrics for inactive cards remain NULL.
- Natural business keys are sufficient.


## 3. gold_merchant_behavior

**Grain:** one row per `merchant_id`

**Rows:** 100,343

Provides merchant-level transaction, spending, fraud, customer, card, MCC, geographic, and online-transaction metrics.

Key characteristics:

- `merchant_id` is not treated as a physical location.
- Merchant → MCC is not assumed to be one-to-one.
- `unique_mccs` is used instead of a single permanent MCC attribute.
- Merchant location remains transaction-level.
- NULL merchant locations are not imputed.
- Amounts use signed values: `SUM(amount)`.


## 4. gold_fraud_features

**Grain:** one row per Silver transaction

**Rows:** 24,386,900

Provides transaction context together with historical fraud features at three business grains:

### User

- 30-day transaction count
- 30-day fraud count
- 30-day fraud rate

### User + Card

- 30-day transaction count
- 30-day fraud count
- 30-day fraud rate

### Merchant

- 30-day transaction count
- 30-day fraud count
- 30-day fraud rate




# Historical Fraud Feature Semantics

All historical fraud features use a strict prior 30-day window:


`historical_timestamp >= current_timestamp - 30 days AND historical_timestamp < current_timestamp`

This ensures:

- the current transaction cannot contribute to its own features
- transactions sharing the current timestamp cannot contribute to one another
- tied timestamps are treated as simultaneous
- no artificial transaction ordering is introduced

The temporary numeric timestamp representation used internally for Spark range windows is not persisted.


# Key Engineering Decisions

The project intentionally follows these principles:

- Bronze and Silver are not modified by Gold development
- Natural/business keys are used where sufficient
- Surrogate keys are not introduced without a genuine requirement
- Signed transaction amounts are preserved
- `SUM(ABS(amount))` is not used
- Merchant IDs are not interpreted as physical locations
- MCC is not assumed to be a permanent merchant attribute
- NULL merchant locations are not artificially imputed
- Transaction ordering is not manufactured
- Historical fraud features use strictly prior timestamps
- Tied timestamps are treated as simultaneous
- Validation is performed against persisted Delta tables, not only in-memory DataFrames




# Validation

Gold objects were individually validated during implementation and subsequently subjected to end-to-end validation.

## Gold v1 E2E Validation

| Check | Result |
|---|---|
| Gold tables exist | PASS |
| Expected row counts | PASS |
| Silver → Gold transaction population | PASS |
| Amount reconciliation | PASS |
| Fraud reconciliation | PASS |
| User grain | PASS |
| Card grain | PASS |
| Merchant grain | PASS |
| Critical NULL checks | PASS |
| Persisted Delta tables | PASS |

## Key Reconciliations

| Metric | Result |
|---|---:|
| Silver transactions | 24,386,900 |
| Gold fraud-feature rows | 24,386,900 |
| Silver total amount | 1,064,098,117.16 |
| Gold total amount | 1,064,098,117.16 |
| Silver fraud count | 29,757 |
| Gold fraud count | 29,757 |

Historical-feature leakage was independently tested for:

- User history
- Card history
- Merchant history

All tests returned **zero mismatches**.

### Overall Status

**GOLD v1 E2E VALIDATION — PASS**



# Power BI Report
The final analytical results are presented through a five-page Power BI report connected directly to the Databricks Gold layer. The report is descriptive and focuses on user behavior, card behavior, merchant behavior, historical fraud features, and the main evidence-based findings.

1. User Behavior
User Behavior — Overview of transaction volume, spending, user-level transaction activity, average transaction amounts, and observed fraud-rate distribution.
2. Card Behavior
Card Behavior — Comparison of transaction volume, spending, average transaction amounts, and transaction-weighted fraud rates across card types.
3. Merchant Behavior
Merchant Behavior — Merchant-level transaction and spending concentration together with online merchant activity.
4. Fraud Features
Fraud Features — Descriptive comparison of historical fraud rates and recent historical fraud-count features associated with observed fraudulent and non-fraudulent transactions.
5. Key Findings
Key Findings — Summary of the strongest descriptive findings from the analytical workflow.

The Power BI report is provided as powerbi/credit_card_analytics.pbix. The report connects directly to the Databricks Gold layer; the underlying 24.4M-row transaction dataset is not stored in the repository.

# Deferred Scope

The following object is intentionally deferred:

`gold_monthly_user_spending`

It is not part of Gold v1 and should only be implemented if project scope is explicitly expanded.

No additional production Gold objects are currently required.



# Final Analytical Status

The Gold v1 data products have been used for descriptive analytical exploration covering:
- user behavior
- card behavior
- merchant behavior
- historical fraud features

The resulting findings are presented in the Power BI report.

No machine-learning models, predictive claims, or causal claims are part of the current project scope.

No additional production Gold tables are currently required.

